# 🎬 Supernan AI Intern – Hindi Dubbing Pipeline
**Google Colab Notebook (Free T4 GPU)**

This notebook runs the full 7-stage pipeline on a 15-second clip of the Supernan training video.

Stages:
1. Install dependencies
2. Clone repo & download video
3. Extract clip + audio
4. Transcribe (Whisper large-v3 auto-detects Kannada)
5. Translate → Hindi (IndicTrans2 indic-indic-1B)
6. Synthesise Hindi voice (Coqui XTTS v2)
7. Lip-sync (VideoReTalking)
8. Face restore (GFPGAN)
9. Assemble & download output

> **Runtime**: Set Runtime → **T4 GPU** before running.
> **Source language**: Kannada (auto-detected). Clip window: **0:45 – 1:00** (confirmed speech).

## Cell 1 – Install System Packages

In [ ]:
!apt-get install -y -q ffmpeg rubberband-cli
!ffmpeg -version | head -1

## Cell 2 – Install Python Dependencies

In [ ]:
!pip install -q \
    ffmpeg-python pydub colorlog \
    openai-whisper \
    deep-translator \
    TTS pyrubberband soundfile \
    facexlib basicsr gfpgan \
    huggingface_hub transformers sentencepiece sacremoses \
    opencv-python-headless
print('✓ Python packages installed')

## Cell 3 – Install IndicTrans2

In [ ]:
!pip install -q git+https://github.com/AI4Bharat/IndicTrans2.git
print('✓ IndicTrans2 installed')

## Cell 4 – Clone VideoReTalking

In [ ]:
import os

if not os.path.exists('video-retalking'):
    !git clone https://github.com/vinthony/video-retalking.git

if not os.path.exists('video-retalking/checkpoints'):
    os.makedirs('video-retalking/checkpoints', exist_ok=True)
    from huggingface_hub import snapshot_download
    snapshot_download(
        'vinthony/video-retalking',
        local_dir='video-retalking/checkpoints',
        ignore_patterns=['*.git*'],
    )

!pip install -q -r video-retalking/requirements.txt
print('✓ VideoReTalking ready')

## Cell 5 – Clone Pipeline Repo

In [ ]:
REPO_URL = 'https://github.com/sudip-kumar-prasad/supernan-hindi-dubbing-pipeline.git'

if not os.path.exists('supernan-hindi-dubbing-pipeline'):
    !git clone {REPO_URL}

import sys
sys.path.insert(0, '/content/supernan-hindi-dubbing-pipeline')
print('✓ Pipeline repo ready')

## Cell 6 – Download Source Video

- **Option A** (default): gdown from Google Drive
- **Option B**: Upload manually via `files.upload()`

In [ ]:
# Option A: Download from Google Drive (shared by Supernan)
!pip install -q gdown
FILE_ID = '1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW'
!gdown {FILE_ID} -O /content/input.mp4

# Option B: manual upload
# from google.colab import files
# files.upload()   # select input.mp4

import os
assert os.path.exists('/content/input.mp4'), 'input.mp4 not found'
print(f'✓ Video ready: {os.path.getsize("/content/input.mp4") / 1e6:.1f} MB')

## Cell 7 – Run the Pipeline

In [ ]:
os.chdir('/content/supernan-hindi-dubbing-pipeline')

# 0:45 – 1:00 window — confirmed Kannada speech; large-v3 for best accuracy
!python dub_video.py \
    --input /content/input.mp4 \
    --output /content/output.mp4 \
    --start 45 \
    --end 60 \
    --model large-v3 \
    --verbose

## Cell 8 – Preview & Download Output

In [ ]:
from IPython.display import Video, display
import os

output_path = '/content/output.mp4'
assert os.path.exists(output_path), 'output.mp4 not found – check logs above'
print(f'Output: {os.path.getsize(output_path) / 1e6:.1f} MB')
display(Video(output_path, embed=True, width=640))

In [ ]:
from google.colab import files
files.download('/content/output.mp4')

---
## 💡 Tips

- The source video is in **Kannada** (detected by Whisper). `large-v3` handles it well.
- Checkpoint resume: if a cell fails, just re-run — completed stages are skipped automatically.
- For full-video processing add `--batch` flag to handle long audio in silence-split chunks.
- If gdown fails (quota/permissions), comment it out and use Option B (manual upload) in Cell 6.